# 数据驱动决策（ACCA 先修课）· Decision Making with Data
## 第 4 周：线性回归分析

**授课教师**：王奇 副教授（四川大学商学院）  
**研究方向**：人工智能与公司金融  
**邮箱**：qiwangphd@scu.edu.cn

---

### 本周学习目标
1. 深入理解最小二乘法的数学原理与正规方程推导
2. 掌握回归系数 $a$、$b$ 的两种等价计算公式
3. 理解相关系数 $r$ 与决定系数 $R^2$ 的含义
4. 用 numpy / scipy / sklearn 三种工具完成回归与预测

In [ ]:
import numpy as np                                # 数值计算
import matplotlib.pyplot as plt                   # 绘图
from scipy import stats                           # scipy 统计模块（linregress）
from sklearn.linear_model import LinearRegression # sklearn 线性回归模型

plt.rcParams['font.sans-serif'] = ['Songti SC', 'SimHei', 'PingFang SC']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False        # 负号显示

## 1. 最小二乘法：手动实现

**目标**：最小化残差平方和 $SSE = \sum(y_i - a - bx_i)^2$

对 $a$、$b$ 求偏导并令其为零，得正规方程，解出：

$$b = \frac{n\sum x_iy_i - \sum x_i \sum y_i}{n\sum x_i^2 - (\sum x_i)^2}, \qquad a = \bar{y} - b\bar{x}$$

In [ ]:
x = np.array([1, 2, 3, 4, 5])                     # 课堂例题自变量
y = np.array([2, 3, 5, 4, 6])                     # 课堂例题因变量
n = len(x)                                        # 样本量 n = 5

sum_x = np.sum(x)                                 # Σx = 15
sum_y = np.sum(y)                                 # Σy = 20
sum_xy = np.sum(x * y)                            # Σxy = 69
sum_x2 = np.sum(x ** 2)                           # Σx² = 55

print(f'n={n}, Σx={sum_x}, Σy={sum_y}, Σxy={sum_xy}, Σx²={sum_x2}')   # 打印各求和项

b = (n * sum_xy - sum_x * sum_y) / (n * sum_x2 - sum_x ** 2)   # 求和形式计算斜率 b
a = np.mean(y) - b * np.mean(x)                   # 截距 a = ȳ - b·x̄
print(f'回归方程：ŷ = {a} + {b}x')                  # 应为 ŷ = 1.3 + 0.9x

In [ ]:
# ===== 离差形式验证：与求和形式完全等价 =====
x_mean, y_mean = np.mean(x), np.mean(y)           # 样本均值 x̄=3, ȳ=4
b_check = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean) ** 2)   # 离差形式：Σ(x-x̄)(y-ȳ)/Σ(x-x̄)²
a_check = y_mean - b_check * x_mean               # 截距公式

print(f'离差形式：b = {b_check:.4f}, a = {a_check:.4f}')   # 应与上面完全一致
print(f'求和形式：b = {b:.4f}, a = {a:.4f}')        # 对比验证

## 2. 相关系数 r 与 决定系数 R²

- $r \in [-1, 1]$：衡量线性关系的**方向与强度**
- $R^2 = r^2 = SSR/SST = 1 - SSE/SST$：模型解释变异的**比例**，恒有 $SST = SSR + SSE$

In [ ]:
y_pred = a + b * x                                # 用回归方程计算预测值 ŷ

SST = np.sum((y - y_mean) ** 2)                   # 总变异：Σ(y-ȳ)²
SSR = np.sum((y_pred - y_mean) ** 2)              # 回归变异：Σ(ŷ-ȳ)²
SSE = np.sum((y - y_pred) ** 2)                   # 残差变异：Σ(y-ŷ)²

R2 = SSR / SST                                    # 决定系数 = SSR/SST

# 手动计算 r（柯西-施瓦茨不等式保证 |r|≤1）
r = np.sum((x - x_mean) * (y - y_mean)) / np.sqrt(np.sum((x - x_mean)**2) * np.sum((y - y_mean)**2))   # Pearson 相关系数

print(f'SST = {SST}, SSR = {SSR:.1f}, SSE = {SSE:.1f}')   # 应满足 SST=SSR+SSE=10
print(f'验证 SST = SSR + SSE: {np.isclose(SST, SSR + SSE)}')   # True 表示恒等式成立
print(f'R² = {R2}')                                 # 0.81
print(f'r = {r:.4f}（正线性相关）')                   # 0.9
print('解读：x 能解释 y 变异的 81%，拟合效果良好')     # R² 的业务解读

## 3. 三种工具完成回归（互相验证）

In [ ]:
# ===== 成本数据：产量 vs 总成本 =====
xc = np.array([100, 150, 200, 250, 300, 350, 400])           # 产量
yc = np.array([12000, 14500, 17000, 19000, 21500, 24000, 26500])   # 总成本

# 方法 1：scipy 快速回归
res = stats.linregress(xc, yc)                    # 一行完成回归
print(f'scipy: a = {res.intercept:.2f}, b = {res.slope:.2f}')   # 截距与斜率
print(f'scipy: r = {res.rvalue:.4f}, R² = {res.rvalue**2:.4f}, p值 = {res.pvalue:.6f}')   # r、R² 与显著性

# 方法 2：sklearn 标准机器学习接口
X = xc.reshape(-1, 1)                             # sklearn 要求 X 为二维数组（n行1列）
model = LinearRegression()                        # 创建模型对象
model.fit(X, yc)                                  # 用数据训练（拟合）模型
print(f'sklearn: a = {model.intercept_:.2f}, b = {model.coef_[0]:.2f}')   # 截距与系数
print(f'sklearn R² = {model.score(X, yc):.4f}')    # score 直接返回 R²

In [ ]:
# ===== 用模型预测未来产量成本 =====
x_new = np.array([[450], [500]])                  # 待预测的产量（必须二维）
predictions = model.predict(x_new)                # 模型预测
print(f'产量 450 件 → 预测成本 {predictions[0]:.0f} 元')   # 打印预测 1
print(f'产量 500 件 → 预测成本 {predictions[1]:.0f} 元')   # 打印预测 2
print('注意：450/500 超出样本范围(100~400)，外推预测风险更大')   # 风险提示

## 4. 可视化与残差分析

In [ ]:
b_m, a_m = np.polyfit(xc, yc, 1)                  # 拟合（numpy 方法）
x_line = np.linspace(50, 500, 100)                # 画线范围扩展到 500
y_line = a_m + b_m * x_line                       # 拟合线 y 值

plt.figure(figsize=(10, 6))                        # 画布
plt.scatter(xc, yc, color='#C49A6C', s=80, zorder=3, label='历史数据')   # 历史散点
plt.plot(x_line, y_line, color='#8B6F4E', linewidth=2, label=f'y={a_m:.0f}+{b_m:.2f}x')   # 拟合线

y_pred_pts = a_m + b_m * np.array([450, 500])     # 预测点 y 值
plt.scatter([450, 500], y_pred_pts, color='red', s=120, marker='*', zorder=4, label='预测点')   # 红色星号标记预测
plt.xlabel('产量（件）')                            # x 轴
plt.ylabel('总成本（元）')                          # y 轴
plt.title('成本回归分析与预测')                      # 标题
plt.legend()                                      # 图例
plt.grid(True, alpha=0.3)                         # 网格
plt.show()                                        # 显示

In [ ]:
# ===== 残差分析：检验模型合理性 =====
yc_pred = a_m + b_m * xc                           # 各点的预测值
residuals = yc - yc_pred                           # 残差 e = 实际值 - 预测值

plt.figure(figsize=(10, 4))                        # 扁平画布适合残差图
plt.scatter(xc, residuals, color='#C49A6C', s=60)  # 残差散点
plt.axhline(y=0, color='#8B6F4E', linestyle='--')  # 零参考线
plt.xlabel('产量 x')                                # x 轴
plt.ylabel('残差')                                  # y 轴
plt.title('残差图（检验随机性：应无规律地散布在零线附近）')   # 标题
plt.grid(True, alpha=0.3)                          # 网格
plt.show()                                         # 显示

print(f'残差均值 = {np.mean(residuals):.4f}（应接近 0）')   # 残差均值检验
print(f'残差标准差 = {np.std(residuals, ddof=1):.2f}')      # 残差波动幅度
print('若残差呈弯月形/喇叭形 → 提示线性假设或方差齐性问题')   # 读图指导

## 5. 本周小结

| 工具 | 优点 | 适用 |
|---|---|---|
| numpy 手动 | 理解公式原理 | 学习、验证 |
| scipy.linregress | 快速，输出 r 和 p 值 | 简单回归 |
| sklearn | 标准 ML 接口，可预测 | 实际建模 |

**决策优先级**：残差随机分布 → 模型可靠；$R^2$ 高 ≠ 因果关系，只说明线性解释力。